# Estimación de costos operativos en minería con Q-Q plots y machine learning

- Artículo completo en la plataforma: https://fuzzyfrog.ai/es/ai-lab/proyectos/mineria/estimacion-costos-mineria-qq-plots-machine-learning/
- Este notebook reconstruye, con datos sintéticos, el mismo flujo del proyecto original: limpieza, validación de distribuciones con Q-Q plots, selección de variables y comparación de seis modelos de regresión.
- **Nota:** el dataset usado es sintético, generado para fines demostrativos, con la misma estructura y comportamiento que los datos originales. No contiene datos reales de la fuente original ni el nombre de la operación minera.

## Diagrama de arquitectura

`Registros diarios → limpieza (errores y outliers) → validación de distribuciones (Q-Q plots) → selección de variables (Random Forest) → división y normalización → comparación de 6 modelos → costo estimado por tonelada`

La validación de distribuciones ocurre antes de tocar cualquier modelo, es la etapa que decide si hace falta transformar una variable.

## Carga de datos

Se carga el dataset sintético de registros diarios de operación minera.

In [ ]:
import pandas as pd

data = pd.read_csv("outputs/dataset_sintetico_costos_mineria.csv", parse_dates=["Fecha"])
print(f"Numero de observaciones: {data.shape[0]}")
print(f"Numero de variables: {data.shape[1] - 1}")
data.head(3)

## Explicación de datos

- `Producción (tn)`: toneladas producidas ese día.
- `Striping Ratio`: relación de desbroce, material removido por tonelada de mineral.
- `Humedad(%)`: humedad del material, registrada en pocos valores enteros dentro del rango observado.
- `Costo (US$/TN)`: variable objetivo, costo operativo por tonelada ese día.

Al igual que en el proyecto original, se detectan registros de costo igual a cero, un error evidente de captura que se filtra antes de continuar.

In [ ]:
print("Valor minimo de costo antes de filtrar:", data["Costo (US$/TN)"].min())
data = data[data["Costo (US$/TN)"] > 0]
data.describe()

## Análisis de datos: validación de distribuciones con Q-Q plots

Un Q-Q plot ordena los valores observados de una variable y los compara contra los cuantiles que esperaría ver si esa variable siguiera una distribución teórica. Si los puntos caen sobre la línea diagonal de referencia, la variable se ajusta bien a esa distribución. Curvas en los extremos indican colas más pesadas o más ligeras de lo esperado.

Se comparan tres distribuciones candidatas para cada variable: normal, gamma y lognormal, para decidir con evidencia si alguna necesita una transformación antes de modelar.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

variables = ['Producción (tn)', 'Striping Ratio', 'Humedad(%)', 'Costo (US$/TN)']

fig, axes = plt.subplots(len(variables), 3, figsize=(15, 5 * len(variables)))

for i, variable in enumerate(variables):
    serie = data[variable]

    # Normal
    stats.probplot(serie, dist="norm", plot=axes[i, 0])
    axes[i, 0].set_title(f"Normal Q-Q Plot de {variable}")

    # Gamma (requiere valores positivos)
    serie_pos = serie[serie > 0]
    shape_gamma, loc_gamma, scale_gamma = stats.gamma.fit(serie_pos)
    stats.probplot(serie_pos, dist=stats.gamma, sparams=(shape_gamma, loc_gamma, scale_gamma), plot=axes[i, 1])
    axes[i, 1].set_title(f"Gamma Q-Q Plot de {variable}")

    # Lognormal (requiere valores positivos)
    shape_ln, loc_ln, scale_ln = stats.lognorm.fit(serie_pos)
    stats.probplot(serie_pos, dist=stats.lognorm, sparams=(shape_ln, loc_ln, scale_ln), plot=axes[i, 2])
    axes[i, 2].set_title(f"Lognormal Q-Q Plot de {variable}")

plt.tight_layout()
plt.show()

**Lectura de los resultados:**

- `Producción (tn)` y `Costo (US$/TN)` se ajustan razonablemente bien a una distribución normal en el cuerpo de la distribución, con cierta desviación en los extremos, algo esperable en datos operativos reales.
- `Striping Ratio` muestra un comportamiento similar, aceptable bajo el supuesto normal para efectos de este modelo.
- `Humedad(%)` no se ajusta bien a ninguna de las tres distribuciones continuas evaluadas. Su Q-Q plot muestra escalones bien definidos en vez de una nube continua de puntos, porque en la práctica se registra en muy pocos valores enteros. Esto se documenta como una característica real del proceso de medición, no como un error a corregir.

## Selección de variables

Se mide la importancia relativa de cada variable con un bosque aleatorio, antes de comprometerse con cualquier modelo final.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

X = data[['Producción (tn)', 'Striping Ratio', 'Humedad(%)']]
y = data['Costo (US$/TN)']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

importancia_df = pd.DataFrame({'Variable': X.columns, 'Importancia': rf.feature_importances_})
importancia_df.sort_values('Importancia', ascending=False)

## Modelado: preparación y división de datos

Se dividen los datos en entrenamiento y prueba, y se normalizan para los modelos que lo requieren.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = scaler_X.fit_transform(X_train.values)
X_test_scaled = scaler_X.transform(X_test.values)
y_train_scaled = scaler_y.fit_transform(y_train.to_numpy().reshape(-1, 1))

## Modelado: comparación de seis modelos de regresión

Se entrenan y comparan regresión lineal, árbol de decisión, bosque aleatorio, SVR, gradient boosting y un perceptrón multicapa, con hiperparámetros afinados por validación cruzada en el proyecto original.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

modelos = {
    "Regresion Lineal": LinearRegression(),
    "Arbol de Decision": DecisionTreeRegressor(max_depth=10, min_samples_leaf=4, min_samples_split=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=4, random_state=42),
    "SVR": SVR(C=1.0, kernel="rbf"),
    "Gradient Boosting": GradientBoostingRegressor(learning_rate=0.05, max_depth=3, n_estimators=100, random_state=42),
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    resultados[nombre] = {
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
    }

pd.DataFrame(resultados).T

## Evaluación

Se comparan las métricas de error de los modelos entrenados, para elegir el más adecuado según el objetivo del negocio, no solo según la métrica más favorable.

In [ ]:
resultados_df = pd.DataFrame(resultados).T.sort_values("MAE")
resultados_df

## Hallazgos principales

- La producción diaria es, por amplio margen, la variable más influyente sobre el costo por tonelada, seguida de la relación de desbroce y, muy por detrás, la humedad.
- Validar las distribuciones con Q-Q plots antes de modelar confirmó que no era necesario transformar el costo antes de usar un modelo lineal como punto de comparación.
- La humedad se comporta como una variable casi categórica, con muy pocos valores únicos, algo que solo se hizo evidente al revisar su Q-Q plot.
- Comparar seis modelos, en vez de uno solo, dio evidencia objetiva para recomendar el modelo final, en lugar de asumir que el primero que funcionó era suficiente.